# Verified same-image alignment — F, hai GPU

Một nhánh trên **GPU T4 ×2**, Internet ON, **Save & Run All**. Bạn tự chạy notebook.
Giữ kiến trúc E và nền caption SCST+CE+thưởng đối tượng; thay loss giả nhãn bằng nhãn V-COCO xác nhận trên cùng ảnh.
Không cần thêm dataset: nhãn và mã đã nhúng sẵn. Giữ sáu Input giống A/E.

503 cặp fit / 310 ảnh; 140 cặp calibration / 79 ảnh; probe cũ 62 cặp / 59 ảnh.
Không dùng nhãn calibration/probe để huấn luyện hay chọn checkpoint. Không dùng test caption.
Tập fit nhỏ và được lặp lại; chưa có bằng chứng phương pháp này cải thiện caption.


In [ ]:
print('VERIFIED SAME IMAGE F: BOOTSTRAP START', flush=True)
from pathlib import Path

ARM = 'F'
CHECKPOINT = Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
LEGACY_PROMPT = Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
UPT_PROMPT = Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2/prompt_cache.json')
VISUAL_CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS = Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT = Path('/kaggle/working/semantic_evidence_verified_pairs_F')

# Bạn của bạn sửa các mount ở trên nếu khác; giữ nguyên nội dung đầu vào.
# Không cần COCO_IMAGES hay raw UPT cache: dùng visual HDF5 + prompt JSON đã có.

# Khởi tạo từ checkpoint nền gốc (SHA 5f8...), KHÔNG dùng checkpoint E/bin.


## 1. Chuẩn bị môi trường và mã nguồn
Kho mã đặt trong `/tmp` để không chiếm danh sách Output. Không cài lại PyTorch của Kaggle.

In [ ]:
import json, os, shutil, subprocess, sys
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', CUBLAS_WORKSPACE_CONFIG=':4096:8',
                  HF_HUB_DISABLE_XET='1', HF_HUB_DOWNLOAD_TIMEOUT='120', TOKENIZERS_PARALLELISM='false',
                  PYTHONHASHSEED='42', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
REPO = Path('/tmp/Image_Captioning-verified-same-image')
CHAIR = Path('/tmp/semantic-evidence-pilot-chair')
SETUP_LOG = Path('/tmp/semantic_evidence_pilot_setup_' + ARM + '.log')
PROJECT_COMMIT = 'c09823178a9b28186e71554d368e2716c22af176'

def run(args, cwd=None, log=None):
    args = list(map(str,args))
    print('\nRunning:', ' '.join(args), flush=True)
    with Path(log or SETUP_LOG).open('a',encoding='utf-8') as output:
        output.write('\nRunning: ' + ' '.join(args) + '\n'); output.flush()
        with subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,errors='replace',bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True); output.write(line); output.flush()
            status = process.wait()
    if status: raise RuntimeError(f'Command failed ({status}); full log: {log or SETUP_LOG}')

def checkout(url,path,revision):
    if not path.exists(): run(['git','clone','--no-checkout',url,path])
    assert (path/'.git').is_dir(),path
    run(['git','fetch','--depth','1','origin',revision],cwd=path)
    run(['git','checkout','--detach',revision],cwd=path)

for file in (CHECKPOINT,LEGACY_PROMPT,UPT_PROMPT,COCO_JSON,DETECTIONS):
    assert file.is_file(),f'Missing exact input: {file}. Copy mounted path from Input.'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),f'Choose a fresh OUTPUT or restart session: {OUTPUT}'
import torch
assert torch.cuda.device_count()==2,'Select GPU T4 x2: ONE notebook, ONE arm, TWO GPUs.'
for i in range(2): print('GPU',i,torch.cuda.get_device_name(i),flush=True)
checkout('https://github.com/Supzxjee/Image_Captioning.git',REPO,PROJECT_COMMIT)
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2',
     'h5py','pandas','Pillow','tqdm','scipy','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
     'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])

import base64, hashlib, zlib
BUNDLE_B64 = '...'
BUNDLE_SHA = '90d12e4bb4311126d16cea772357f644f8374cb37d95a3a501a41e97457f7055'
bundle_bytes = zlib.decompress(base64.b64decode(BUNDLE_B64,validate=True))
assert hashlib.sha256(bundle_bytes).hexdigest()==BUNDLE_SHA,'Embedded bundle changed'
bundle = json.loads(bundle_bytes)
SOURCES = bundle['sources']
VERIFIED_LABELS = bundle['labels']
SOURCE_HASHES = {'pilot_model.py': '88cb1f4587b88d1400ab05ac7aa8f8f0535a9e16c698f60036270290df9a3d59', 'run_verified.py': '3e3be190b8b3e337efd1357824dcaa7e30f75ce59227d9c0d78bac0a579d918f', 'verified_data.py': '3b197271e103fb7f5b8513b02f4152b972136f4f243e13e829ff0395f9521878', 'test_verified_data.py': '79b5cab158c9ce5658eeaab0c7b0540df6b447ba42b2bfc6f5bb3e91ffb95a20', 'test_verified_model.py': '02f0bda7591229022df3b379dce6fedb9533f139bc0a8a76b45828794482bc01', 'test_verified_ddp.py': 'cb4884a036b736268cccd51bea0eb92e6818a731bf25a1c24094bf77c65ed6a6', 'pilot_data.py': 'aac539b7c23f4acc3ac05b99496ef8eb79f740e6c99d11742b6de08fd2b892c0', 'test_pilot_data.py': '5481c322e2266447028d93c694d765b3dbefb77428d6a9a9eea31d75422e1f8f', 'pilot_metric_smoke.py': '1f33a26f3edcd21574189464633ac2060ac8cff0a69ec0af27505b07cecbadf6', 'evidence_core.py': '4f8189de7d183bb062247043d6a2ce7bcb6fbd9fb7d0d485922c065d8cbe962b', 'base_runner.py': 'b8a58ecdccf89151aedbedad3ca0ad73832438db8bae707cdcf1aa5800830801', 'probe_data.py': 'f8dfb24cf825ecc2935bfc062731009197459cd55ebb9b0cbb146d248d574e9f', 'synthetic_fixtures.py': '44671927a64ed6d9d0c3a09137fcf9e680641ebba64e29d1d233945026ea1988'}
VERIFIED_MANIFEST_SHA = '7242a9a632ecf715d47cb2a9838a4ecacfad06b9eeca9849f6aa7ebc9e4beffa'
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest()==SOURCE_HASHES[name],name
    (REPO/name).write_text(source,encoding='utf-8')
assert hashlib.sha256(VERIFIED_LABELS.encode()).hexdigest()==VERIFIED_MANIFEST_SHA
VERIFIED_MANIFEST = REPO/'verified_manifest.json'
VERIFIED_MANIFEST.write_text(VERIFIED_LABELS,encoding='utf-8')
print('Verified F source and locked labels installed',flush=True)


## 2. Preflight
Kiểm tra dữ liệu, loss và gradient bằng dữ liệu giả, sau đó đồng bộ hai GPU và metric. Notebook dừng nếu kiểm tra lỗi.


In [ ]:
for name in ('test_pilot_data.py','test_verified_data.py','test_verified_model.py'):
    run([sys.executable,'-u',REPO/name],cwd=REPO)
run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
     '--max_restarts=0',REPO/'test_verified_ddp.py'],cwd=REPO)
run([sys.executable,'-u',REPO/'pilot_metric_smoke.py',CHAIR],cwd=REPO)
print('PREFLIGHT PASS',flush=True)


## 3. Chuẩn bị dữ liệu
Stage cùng 11.000 ảnh caption như A/E và các ảnh trong bộ nhãn xác nhận.
Embedding CLIP đóng băng; riêng tập fit được đưa vào loss. Hai tập còn lại chỉ dùng chẩn đoán.


In [ ]:
CONFIG = Path('/tmp/semantic_evidence_pilot_config_' + ARM + '.json')
expected = {
    'checkpoint':'5f8eead3f4e960189345b1a9bf492763fe52ec71f70115b73244ffe9d03d2e75',
    'legacy_prompt':'d7910e9bf87e3aad86d806c77ec336b34793e3bef68a3f662fdcb2e9456ca38d',
    'upt_prompt':'24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d',
    'dataset_json':'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d',
    'detections':'498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c',
}
cfg = dict(arm=ARM,checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),upt_prompt=str(UPT_PROMPT),
           visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),detections=str(DETECTIONS),
           output=str(OUTPUT),stage='/tmp/semantic_evidence_stage_'+ARM+'.h5',chair=str(CHAIR),
           global_batch_size=8,eval_batch_size=16,warmup_updates=200,base_lr=5e-6,branch_lr=1e-4,
           scst_weight=.05,expected_hashes=expected,seed=42,project_commit=PROJECT_COMMIT,
           embedded_source_sha256=SOURCE_HASHES,verified_manifest=str(VERIFIED_MANIFEST),
           verified_manifest_sha256=VERIFIED_MANIFEST_SHA,aux_stage='/tmp/verified_same_image_aux_F.h5')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
run([sys.executable,'-u',REPO/'run_verified.py','prepare','--config',CONFIG],cwd=REPO)
PROTOCOL = OUTPUT/'protocol.json'
(OUTPUT/'source').mkdir()
for name,source in SOURCES.items(): (OUTPUT/'source'/name).write_text(source,encoding='utf-8')
shutil.copy2(SETUP_LOG,OUTPUT/'setup.log')
locked = json.loads(PROTOCOL.read_text())
print('LOCKED COHORT:',locked['cohort_sha256'],locked['counts_by_mode'],flush=True)

print('LOCKED VERIFIED COUNTS:',locked['verified_counts'],flush=True)


## 4. Huấn luyện F trên hai GPU
1.250 cập nhật; batch toàn cục 8, mỗi GPU 4 ảnh caption + 4 cặp xác nhận.
200 cập nhật đầu: CE + 0,1 loss căn chỉnh. Sau đó thêm 0,05 SCST với reward CIDEr-D + 0,5 thưởng đối tượng.
Loss căn chỉnh: max(0; 0,2 − điểm đúng + điểm sai), dùng cặp đã xác nhận trên cùng ảnh.
Lịch cố định 5.000 lượt sai quan hệ và 5.000 lượt sai người; không dùng loss giả nhãn E cũ.
Giữ checkpoint cuối; không chọn bằng calibration/probe/test. Đây là nhiều giám sát và tính toán hơn A/E.


In [ ]:
TRAIN_LOG = OUTPUT/'run.log'
try:
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
         '--max_restarts=0',REPO/'run_verified.py','worker','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
except Exception:
    run([sys.executable,'-u',REPO/'run_verified.py','archive','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
    raise


## 5. Đánh giá và lưu
Caption greedy val1000, cùng A/E; thêm tắt nhánh, tráo text, tráo vùng.
Matcher trước/sau học trên calibration và probe cố định; có toàn ảnh và xoá đặc trưng làm kiểm tra đối chứng.
Calibration/probe thuộc caption train/restval và đã từng dùng trong nghiên cứu trước, không phải test hoàn toàn mới.
Sai vật thể chỉ kiểm tra chuyển giao, không có cặp âm vật thể được tự suy diễn để học.
Gửi ZIP review; giữ riêng checkpoint.pth trong Saved Output. ZIP không chứa checkpoint/HDF5.


In [ ]:
try:
    run([sys.executable,'-u',REPO/'run_verified.py','evaluate','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
finally:
    run([sys.executable,'-u',REPO/'run_verified.py','archive','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
result = json.loads((OUTPUT/'run_summary.json').read_text())
print(json.dumps(result,indent=2,ensure_ascii=False),flush=True)
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.pth',flush=True)
